# Cost Tables

This notebook builds extractable cost tables as plain pandas DataFrames and exports them to `output/cost_tables`.

In [ ]:
from pathlib import Path
import sys

import pandas as pd
from pint import Quantity


# Make `src` importable whether the notebook is opened from repo root or `nbs/`
root = Path.cwd().resolve()
while root != root.parent and not (root / "src").exists():
    root = root.parent
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

from src.units import Units as U
import src.assumptions as A

pd.options.display.float_format = "{:,.3f}".format

OUTPUT_DIR = root / "output" / "cost_tables"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


GAS_CO2_INTENSITY_CCGT = 0.34 * U.t / U.MWh
GAS_CO2_INTENSITY_GT = 0.50 * U.t / U.MWh
H_DAC_MWH_PER_T = 1.1389 * U.MWh / U.t
DAC_COST_PER_T = 600 * U.GBP / U.t
HOURS_PER_YEAR = A.HoursPerYear
EDAC_ELEC_PER_T = 1.25 * U.MWh / U.t


tech_params = pd.DataFrame(
    {
        "eta_elec": [0.55, 0.325, 0.55],
        "waste_frac": [0.45, 0.675, 0.45],
        "f_lowT": [1.0, 1.0, 1.0],
        "waste_temp_C": [171, 538, 171],
        "dac_elec_per_t": [
            A.LTDAC.ElectricityPerTonCO2.to(U.MWh / U.t),
            A.LTDAC.ElectricityPerTonCO2.to(U.MWh / U.t),
            EDAC_ELEC_PER_T,
        ],
        "co2_intensity": [
            GAS_CO2_INTENSITY_CCGT,
            GAS_CO2_INTENSITY_GT,
            GAS_CO2_INTENSITY_CCGT,
        ],
    },
    index=[
        "CCGT (normal combined cycle)",
        "Gas Turbine (simple cycle)",
        "CCGT with E-DAC",
    ],
)


def annual_generation_from_capacity(capacity_gw: float, capacity_factor: float) -> Quantity:
    """Convert GW and capacity factor to annual generation in TWh."""
    return (capacity_gw * U.GW * HOURS_PER_YEAR * capacity_factor).to(U.TWh)


def compute_metrics(gen_twh: dict[str, float], edac_name: str | None = None) -> pd.DataFrame:
    """Calculate generation, emissions, DAC capture, electricity use, and DAC cost."""
    rows = []

    for tech, gen_val in gen_twh.items():
        params = tech_params.loc[tech]

        eff = params["eta_elec"] * U.dimensionless
        waste_frac = params["waste_frac"] * U.dimensionless
        f_low = params["f_lowT"] * U.dimensionless
        dac_elec = params["dac_elec_per_t"]

        elec = gen_val * U.TWh
        fuel_input = elec / eff
        waste_heat = fuel_input * waste_frac
        usable_heat = waste_heat * f_low
        gross_emissions_t = (elec.to(U.MWh) * params["co2_intensity"]).to(U.t)

        if edac_name is not None and tech == edac_name:
            usable_heat = 0 * U.MWh
            capture_t = (elec.to(U.MWh) / dac_elec).to(U.t)
        else:
            capture_t = (usable_heat.to(U.MWh) / H_DAC_MWH_PER_T).to(U.t)

        dac_elec_mwh = (capture_t * dac_elec).to(U.MWh)
        net_emissions_t = gross_emissions_t - capture_t
        elec_surplus = elec.to(U.MWh) - dac_elec_mwh
        dac_cost = (capture_t * DAC_COST_PER_T).to(U.GBP)

        rows.append(
            {
                "Technology": tech,
                "Generation_TWh": elec.to(U.TWh).magnitude,
                "Fuel_input_TWh": fuel_input.to(U.TWh).magnitude,
                "Waste_heat_TWh": waste_heat.to(U.TWh).magnitude,
                "Usable_lowT_heat_TWh": usable_heat.to(U.TWh).magnitude,
                "Gas_emissions_MtCO2": gross_emissions_t.to(U.Mt).magnitude,
                "DAC_capture_MtCO2": capture_t.to(U.Mt).magnitude,
                "Net_emissions_MtCO2": net_emissions_t.to(U.Mt).magnitude,
                "DAC_electricity_TWh": dac_elec_mwh.to(U.TWh).magnitude,
                "Elec_surplus_after_DAC_TWh": elec_surplus.to(U.TWh).magnitude,
                "DAC_cost_billion_GBP": (dac_cost / (1e9 * U.GBP)).magnitude,
            }
        )

    return pd.DataFrame(rows).set_index("Technology").sort_index()


def q_value(quantity: Quantity, units) -> float:
    """Return a quantity magnitude in the requested units."""
    return quantity.to(units).magnitude


In [ ]:
# Plain DataFrames: these are easy to copy, save, or reuse elsewhere.

cost_input_summary = pd.DataFrame(
    [
        {
            "Parameter": "Heat required for LT-DAC capture",
            "Symbol": "H_DAC_MWH_PER_T",
            "Value": H_DAC_MWH_PER_T.to(U.MWh / U.t).magnitude,
            "Units": "MWh/tCO2",
            "Used for": "LT-DAC capture = usable low-T heat / H_DAC_MWH_PER_T",
        },
        {
            "Parameter": "DAC capture cost",
            "Symbol": "DAC_COST_PER_T",
            "Value": DAC_COST_PER_T.to(U.GBP / U.t).magnitude,
            "Units": "GBP/tCO2",
            "Used for": "DAC cost = captured CO2 * DAC_COST_PER_T",
        },
        {
            "Parameter": "CCGT CO2 intensity",
            "Symbol": "GAS_CO2_INTENSITY_CCGT",
            "Value": GAS_CO2_INTENSITY_CCGT.to(U.t / U.MWh).magnitude,
            "Units": "tCO2/MWh",
            "Used for": "Gross emissions for CCGT cases",
        },
        {
            "Parameter": "GT CO2 intensity",
            "Symbol": "GAS_CO2_INTENSITY_GT",
            "Value": GAS_CO2_INTENSITY_GT.to(U.t / U.MWh).magnitude,
            "Units": "tCO2/MWh",
            "Used for": "Gross emissions for simple-cycle GT",
        },
        {
            "Parameter": "Hours per year",
            "Symbol": "HOURS_PER_YEAR",
            "Value": HOURS_PER_YEAR,
            "Units": "h/yr",
            "Used for": "GW -> TWh conversion",
        },
    ]
)

tech_summary = pd.DataFrame(
    {
        "Electrical efficiency (%)": tech_params["eta_elec"] * 100,
        "Waste-heat fraction (%)": tech_params["waste_frac"] * 100,
        "Low-T usable fraction (%)": tech_params["f_lowT"] * 100,
        "Waste-heat temperature (C)": tech_params["waste_temp_C"],
        "DAC electricity (MWh/tCO2)": [x.to(U.MWh / U.t).magnitude for x in tech_params["dac_elec_per_t"]],
        "Gas CO2 intensity (tCO2/MWh)": [x.to(U.t / U.MWh).magnitude for x in tech_params["co2_intensity"]],
    },
    index=tech_params.index,
)

scenario_generation_twh = {
    "CCGT (normal combined cycle)": 588,
    "Gas Turbine (simple cycle)": 588,
    "CCGT with E-DAC": 588,
}

energy_cost_input_summary = pd.DataFrame(
    [
        {
            "Parameter": "Electricity demand in 2050",
            "Symbol": "A.EnergyDemand2050",
            "Value": q_value(A.EnergyDemand2050, U.TWh),
            "Units": "TWh",
            "Used for": "Final division in energy_cost = total_system_cost / energy_demand",
        },
        {
            "Parameter": "Hours per year",
            "Symbol": "A.HoursPerYear",
            "Value": q_value(A.HoursPerYear, U.h),
            "Units": "h/yr",
            "Used for": "yearly_cost = capacity * capacity_factor * hours_per_year * lcoe",
        },
        {
            "Parameter": "Weighted renewable capacity factor",
            "Symbol": "A.Renewables.AverageCapacityFactor",
            "Value": A.Renewables.AverageCapacityFactor,
            "Units": "fraction",
            "Used for": "Renewables yearly_cost",
        },
        {
            "Parameter": "Weighted renewable LCOE",
            "Symbol": "A.Renewables.AverageLCOE",
            "Value": q_value(A.Renewables.AverageLCOE, U.GBP / U.MWh),
            "Units": "GBP/MWh",
            "Used for": "Renewables yearly_cost",
        },
        {
            "Parameter": "Nuclear capacity",
            "Symbol": "A.Nuclear.Capacity",
            "Value": q_value(A.Nuclear.Capacity, U.GW),
            "Units": "GW",
            "Used for": "Nuclear yearly_cost",
        },
        {
            "Parameter": "Nuclear capacity factor",
            "Symbol": "A.Nuclear.CapacityFactor",
            "Value": A.Nuclear.CapacityFactor,
            "Units": "fraction",
            "Used for": "Nuclear yearly_cost",
        },
        {
            "Parameter": "Weighted nuclear LCOE",
            "Symbol": "A.Nuclear.AverageLCOE",
            "Value": q_value(A.Nuclear.AverageLCOE, U.GBP / U.MWh),
            "Units": "GBP/MWh",
            "Used for": "Nuclear yearly_cost",
        },
        {
            "Parameter": "Hydrogen cavern annualised cost",
            "Symbol": "A.HydrogenStorage.CavernStorage.AnnualisedCost",
            "Value": q_value(A.HydrogenStorage.CavernStorage.AnnualisedCost, U.GBP / U.MWh),
            "Units": "GBP/MWh_storage",
            "Used for": "Storage cost = storage_capacity * annualised_cost",
        },
        {
            "Parameter": "Electrolyser annualised cost",
            "Symbol": "A.HydrogenStorage.Electrolysis.AnnualisedCost",
            "Value": q_value(A.HydrogenStorage.Electrolysis.AnnualisedCost, U.GBP / U.kW),
            "Units": "GBP/kW",
            "Used for": "Storage cost = electrolyser_power * annualised_cost",
        },
        {
            "Parameter": "Hydrogen generation annualised cost",
            "Symbol": "A.HydrogenStorage.Generation.AnnualisedCost",
            "Value": q_value(A.HydrogenStorage.Generation.AnnualisedCost, U.GBP / U.kW),
            "Units": "GBP/kW",
            "Used for": "Storage cost = generation_capacity * annualised_cost",
        },
        {
            "Parameter": "Additional system costs",
            "Symbol": "A.AdditionalCosts",
            "Value": q_value(A.AdditionalCosts, U.GBP / U.MWh),
            "Units": "GBP/MWh",
            "Used for": "additional_costs = A.AdditionalCosts * energy_demand",
        },
        {
            "Parameter": "Dispatchable gas CCS LCOE",
            "Symbol": "A.DispatchableGasCCS.LCOE",
            "Value": q_value(A.DispatchableGasCCS.LCOE, U.GBP / U.MWh),
            "Units": "GBP/MWh",
            "Used for": "Operational gas-CCS cost when sim_df is included",
        },
        {
            "Parameter": "Medium-term storage LCOE",
            "Symbol": "A.MediumTermStorage.LCOE",
            "Value": q_value(A.MediumTermStorage.LCOE, U.GBP / U.MWh),
            "Units": "GBP/MWh",
            "Used for": "Operational medium-storage cost when sim_df is included",
        },
    ]
)

renewable_weighting_inputs = pd.DataFrame(
    [
        {
            "Technology": "Solar",
            "Capacity ratio": A.Renewables.CapacityRatios.Solar,
            "Capacity factor": A.Renewables.CapacityFactors.Solar,
            "LCOE (GBP/MWh)": q_value(A.Renewables.LCOE.Solar, U.GBP / U.MWh),
        },
        {
            "Technology": "Offshore wind",
            "Capacity ratio": A.Renewables.CapacityRatios.OffshoreWind,
            "Capacity factor": A.Renewables.CapacityFactors.OffshoreWind,
            "LCOE (GBP/MWh)": q_value(A.Renewables.LCOE.OffshoreWind, U.GBP / U.MWh),
        },
        {
            "Technology": "Onshore wind",
            "Capacity ratio": A.Renewables.CapacityRatios.OnshoreWind,
            "Capacity factor": A.Renewables.CapacityFactors.OnshoreWind,
            "LCOE (GBP/MWh)": q_value(A.Renewables.LCOE.OnshoreWind, U.GBP / U.MWh),
        },
    ]
)

nuclear_weighting_inputs = pd.DataFrame(
    [
        {
            "Technology": "Existing nuclear",
            "Capacity ratio": A.Nuclear.CapacityRatios.Existing,
            "LCOE (GBP/MWh)": q_value(A.Nuclear.LCOE.Existing, U.GBP / U.MWh),
        },
        {
            "Technology": "Large reactors",
            "Capacity ratio": A.Nuclear.CapacityRatios.LargeReactors,
            "LCOE (GBP/MWh)": q_value(A.Nuclear.LCOE.LargeReactors, U.GBP / U.MWh),
        },
        {
            "Technology": "Small reactors",
            "Capacity ratio": A.Nuclear.CapacityRatios.SmallReactors,
            "LCOE (GBP/MWh)": q_value(A.Nuclear.LCOE.SmallReactors, U.GBP / U.MWh),
        },
    ]
)

scenario_energy_cost_inputs = pd.DataFrame(
    [
        {
            "Scenario input": "renewable_capacity",
            "Typical units": "GW",
            "Used in": "yearly_cost for renewables",
            "Notes": "Set by the scenario or power-system object",
        },
        {
            "Scenario input": "storage_capacity",
            "Typical units": "TWh",
            "Used in": "total_storage_cost",
            "Notes": "Hydrogen cavern storage size",
        },
        {
            "Scenario input": "electrolyser_power",
            "Typical units": "GW",
            "Used in": "total_storage_cost",
            "Notes": "Hydrogen electrolysis capacity",
        },
        {
            "Scenario input": "generation_capacity",
            "Typical units": "GW",
            "Used in": "total_storage_cost",
            "Notes": "Hydrogen-to-power generation capacity",
        },
        {
            "Scenario input": "annual_gas_ccs_energy",
            "Typical units": "TWh/yr",
            "Used in": "Operational gas-CCS cost",
            "Notes": "Only used when simulation outputs are included",
        },
        {
            "Scenario input": "annual_medium_storage_energy",
            "Typical units": "TWh/yr",
            "Used in": "Operational medium-storage cost",
            "Notes": "Only used when simulation outputs are included",
        },
    ]
)

results = compute_metrics(scenario_generation_twh, edac_name="CCGT with E-DAC")

cost_input_summary


In [ ]:
energy_cost_input_summary


In [ ]:
renewable_weighting_inputs


In [ ]:
nuclear_weighting_inputs


In [ ]:
scenario_energy_cost_inputs


In [ ]:
tech_summary


In [ ]:
results


In [ ]:
# Export each table to CSV and to a single Excel workbook.

cost_input_summary.to_csv(OUTPUT_DIR / "cost_input_summary.csv", index=False)
energy_cost_input_summary.to_csv(OUTPUT_DIR / "energy_cost_input_summary.csv", index=False)
renewable_weighting_inputs.to_csv(OUTPUT_DIR / "renewable_weighting_inputs.csv", index=False)
nuclear_weighting_inputs.to_csv(OUTPUT_DIR / "nuclear_weighting_inputs.csv", index=False)
scenario_energy_cost_inputs.to_csv(OUTPUT_DIR / "scenario_energy_cost_inputs.csv", index=False)
tech_summary.to_csv(OUTPUT_DIR / "technology_assumptions.csv")
results.to_csv(OUTPUT_DIR / "gas_tech_cost_results.csv")

with pd.ExcelWriter(OUTPUT_DIR / "cost_tables.xlsx") as writer:
    cost_input_summary.to_excel(writer, sheet_name="cost_inputs", index=False)
    energy_cost_input_summary.to_excel(writer, sheet_name="energy_cost_inputs", index=False)
    renewable_weighting_inputs.to_excel(writer, sheet_name="renewable_inputs", index=False)
    nuclear_weighting_inputs.to_excel(writer, sheet_name="nuclear_inputs", index=False)
    scenario_energy_cost_inputs.to_excel(writer, sheet_name="scenario_inputs", index=False)
    tech_summary.to_excel(writer, sheet_name="tech_assumptions")
    results.to_excel(writer, sheet_name="results")

print(f"Saved extractable tables to: {OUTPUT_DIR}")
